In [1]:
import jax
jax.config.update("jax_enable_x64", True)     # MPS overlaps at float32 are not worth checking
import jax.numpy as jnp
from jax import lax
import numpy as np                            # sector bookkeeping and printing only
import time
from functools import partial
from itertools import combinations

import pyblock3.algebra.ad as ad
ad.ENABLE_JAX = True                          # MUST precede the next three imports
from pyblock3.algebra.ad.core import SparseTensor, SubTensor
from pyblock3.algebra.ad.mps import MPS
from pyblock3.algebra.symmetry import SZ

np.set_printoptions(precision=4, suppress=True, linewidth=110)
print(f"jax {jax.__version__} | {jax.devices()} | pyblock3 ad backend jax: {ad.ENABLE_JAX}")

jax 0.11.1 | [CpuDevice(id=0)] | pyblock3 ad backend jax: True


In [2]:
def ssh_hamiltonian(n, delta, t=1.0):
    """Dimerised chain: t1 = -t(1+delta/2), t2 = -t(1-delta/2)."""
    i = jnp.arange(n - 1)
    off = -t * jnp.where(i % 2 == 0, 1 + delta/2, 1 - delta/2)
    return jnp.diag(off, 1) + jnp.diag(off, -1)

def ssh_orbitals(n, Nf, delta=0.4):
    """Occupied orbitals C (n x Nf): the lowest Nf levels."""
    _, Cf = jnp.linalg.eigh(ssh_hamiltonian(n, delta))
    return Cf[:, :Nf]

n, Nf = 10, 4
C = ssh_orbitals(n, Nf)
Lam = C @ C.T                                  # Lam_ij = <a_i^dag a_j>
print(f"n = {n} sites, Nf = {Nf} fermions, C {C.shape}, orthonormal columns: "
      f"{np.allclose(C.T @ C, jnp.eye(Nf))}")
print(f"Lambda is a projector: max |Lam^2 - Lam| = {jnp.abs(Lam @ Lam - Lam).max():.2e}")

n = 10 sites, Nf = 4 fermions, C (10, 4), orthonormal columns: True
Lambda is a projector: max |Lam^2 - Lam| = 3.33e-16


In [ ]:
#Getting the right angles Theta to zero the correlation matrix
def rotate_rows(U, p, theta):
    """Apply V(theta)^T to rows p, p+1.  Returns a new array (JAX is immutable)."""
    c, s = jnp.cos(theta), jnp.sin(theta)
    rp, rq = U[p], U[p+1]
    return U.at[p].set(c*rp + s*rq).at[p+1].set(-s*rp + c*rq)

def zeroing_angles(v, k):
    """The B-1 angles that rotate v onto its first entry, as one lax.scan."""
    def step(v, j):
        theta = jnp.arctan2(v[j], v[j-1])
        c, s = jnp.cos(theta), jnp.sin(theta)
        return v.at[j-1].set(c*v[j-1] + s*v[j]).at[j].set(0.0), theta
    js = jnp.arange(v.shape[0] - 1, 0, -1)
    _, th = lax.scan(step, v, js)
    return np.asarray(k + js - 1), th

def gmps(C, eps=1e-10, Bmax=None):
    """Fishman-White: orbitals -> (occupations, Givens gates, block sizes)."""
    U, n = jnp.asarray(C, float), C.shape[0]
    Bmax = Bmax or n
    occ, gates, Bs = np.zeros(n, int), [], []
    for k in range(n - 1):
        Lam = U @ U.T                          # must be recomputed: U is rotated every step
        for B in range(2, min(Bmax, n - k) + 1):
            nb, W = jnp.linalg.eigh(Lam[k:k+B, k:k+B])
            if float(jnp.minimum(nb[0], 1.0 - nb[-1])) < eps:
                break
        if nb[0] <= 1.0 - nb[-1]: v, occ[k] = W[:, 0], 0      # nearest empty
        else:                     v, occ[k] = W[:, -1], 1      # nearest filled
        Bs.append(B)
        for p, theta in zip(*zeroing_angles(v, k)):
            U = rotate_rows(U, int(p), theta)
            gates.append((int(p), theta))
    occ[n-1] = int(round(float(U[n-1] @ U[n-1])))
    assert occ.sum() == C.shape[1], f"particle number lost: {occ.sum()} != {C.shape[1]}"
    return occ, gates, Bs, np.asarray(U)      # U is now the ROTATED orbital matrix

occ, gates, Bs, U_rot = gmps(C)
print(f"occupations n_k : {occ}   (sum {occ.sum()} = Nf)")
print(f"block sizes B   : {Bs}")
print(f"gates           : {len(gates)} = sum(B-1);  chi should come out 2**(B-1) = "
      f"{2**(max(Bs)-1)}")

occupations n_k : [0 0 0 1 0 1 1 0 0 1]   (sum 4 = Nf)
block sizes B   : [5, 5, 5, 4, 4, 3, 3, 2, 2]
gates           : 24 = sum(B-1);  chi should come out 2**(B-1) = 16


In [ ]:
#Converting the angles as gates at the MPS level and applying those gates
@jax.jit
def V_hat(theta):
    """Two-site gate, Eq. (9), as [out_p, out_q, in_p, in_q]."""
    c, s = jnp.cos(theta), jnp.sin(theta)
    g = jnp.eye(4).at[1,1].set(c).at[1,2].set(s).at[2,1].set(-s).at[2,2].set(c)
    return g.reshape(2, 2, 2, 2)

@jax.jit
def _gated_pair(Al, Ar, theta):
    T = jnp.tensordot(Al, Ar, axes=1)                      # (Dl, 2, 2, Dr)
    return jnp.einsum("xypq,apqb->axyb", V_hat(theta), T)

def product_state(occ):
    """|n_1 ... n_n> as tensors + the particle number carried by each bond."""
    tensors, qn = [], [np.zeros(1, int)]
    for o in occ:
        tensors.append(jax.nn.one_hot(int(o), 2).reshape(1, 2, 1))
        qn.append(qn[-1] + int(o))
    return tensors, qn

def split_two_site(T, ql, qr, chi_max=None, cutoff=1e-12):
    """SVD-split a (Dl, 2, 2, Dr) tensor inside each particle-number sector.

    Rows carry charge q_l + n_p, columns q_r - n_p; only equal charges connect, which
    hands back the bond charge of every retained singular vector for free.
    """
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl*2, 2*Dr)
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()         # charge of each row
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()         # charge of each column
    sectors = []
    for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        u, s, vt = jnp.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
        sectors.append((nm, r, c, u, s, vt))
    alls = np.concatenate([np.asarray(x[4]) for x in sectors])
    thr = cutoff * max(alls.max(), 1e-300)
    if chi_max is not None and int((alls > thr).sum()) > chi_max:
        thr = np.sort(alls)[::-1][:chi_max][-1]                # one global cap across sectors
    Acols, Brows, qm = [], [], []
    for nm, r, c, u, s, vt in sectors:
        k = int((np.asarray(s) >= thr).sum())
        if k == 0:
            continue
        Acols.append(jnp.zeros((Dl*2, k)).at[r].set(u[:, :k]))
        Brows.append(jnp.zeros((k, 2*Dr)).at[:, c].set(s[:k, None] * vt[:k]))
        qm += [nm] * k
    return (jnp.concatenate(Acols, axis=1).reshape(Dl, 2, -1),
            jnp.concatenate(Brows, axis=0).reshape(-1, 2, Dr), np.array(qm, int))

def apply_gate(tensors, qn, p, theta, chi_max=None, cutoff=1e-12):
    """Gate on (p, p+1), then the sector-blocked split.  Touches one bond only."""
    T = _gated_pair(tensors[p], tensors[p+1], theta)
    tensors, qn = list(tensors), list(qn)
    tensors[p], tensors[p+1], qn[p+1] = split_two_site(
        T, qn[p], qn[p+2], chi_max, cutoff)
    return tensors, qn

def build(occ, gates, chi_max=None, cutoff=1e-12):
    tensors, qn = product_state(occ)
    for p, theta in reversed(gates):                           # reverse of derivation order
        tensors, qn = apply_gate(tensors, qn, p, theta, chi_max, cutoff)
    return tensors, qn

def to_dense(tensors):
    """Contract to a 2**n vector (site 0 = most significant bit), for checking."""
    psi = tensors[0][0]
    for A in tensors[1:]:
        psi = jnp.tensordot(psi, A, axes=([-1], [0]))
    return psi.reshape(-1)

tensors, qn = build(occ, gates, cutoff=1e-14)
print("bond dimensions :", [t.shape[0] for t in tensors] + [tensors[-1].shape[2]])
print("charges on bond 4:", qn[4])
print("tensor backend  :", type(tensors[0]).__name__, tensors[0].dtype)

bond dimensions : [1, 2, 4, 8, 16, 16, 16, 8, 4, 2, 1]
charges on bond 4: [0 1 1 1 1 2 2 2 2 2 2 3 3 3 3 4]
tensor backend  : ArrayImpl float64


In [ ]:
#Creating a pyblock3 MPS
Q = lambda m: SZ(int(m), int(m), 0)            # spinless mode = spin-polarised electron

def to_pyblock3(tensors, qn):
    """Dense tensors + per-bond charges -> pyblock3 block-sparse MPS (jax blocks)."""
    out = []
    for i, A in enumerate(tensors):
        blocks = []
        for nl in sorted(set(qn[i].tolist())):
            rows = np.where(qn[i] == nl)[0]
            for ph in (0, 1):
                cols = np.where(qn[i+1] == nl + ph)[0]
                if len(cols) == 0:
                    continue
                data = A[np.ix_(rows, [ph], cols)]
                if float(jnp.abs(data).max()) == 0.0:
                    continue
                blocks.append(SubTensor(data=data, q_labels=(Q(nl), Q(ph), Q(nl + ph))))
        out.append(SparseTensor(blocks=blocks))
    return MPS(tensors=out)

mps = to_pyblock3(tensors, qn)
print(f"{type(mps).__module__}.{type(mps).__name__}")
print("bond dims:", mps.show_bond_dims(), " norm:", mps.norm())
print(f"\nsite 3 has {len(mps[3].blocks)} blocks; the first three:")
for b in mps[3].blocks[:3]:
    print("   ", b.q_labels, np.asarray(b.data).shape, type(b.data).__name__)

pyblock3.algebra.ad.mps.MPS
bond dims: 1|2|4|8|16|16|16|8|4|2|1  norm: 1.0000000000000016

site 3 has 8 blocks; the first three:
    (< N=0 SZ=0 PG=0 >, < N=0 SZ=0 PG=0 >, < N=0 SZ=0 PG=0 >) (1, 1, 1) ArrayImpl
    (< N=0 SZ=0 PG=0 >, < N=1 SZ=1/2 PG=0 >, < N=1 SZ=1/2 PG=0 >) (1, 1, 4) ArrayImpl
    (< N=1 SZ=1/2 PG=0 >, < N=0 SZ=0 PG=0 >, < N=1 SZ=1/2 PG=0 >) (3, 1, 4) ArrayImpl


In [ ]:
#Computing and checking the sign convention. The sign is given be det(U[occ_rows,:]), where U is the product of every rotations 
def jw_tables(n):
    """Jordan-Wigner gather tables: every a_i / a_i^dag becomes one indexing operation."""
    j = jnp.arange(2**n)
    w = 1 << jnp.arange(n-1, -1, -1)
    bits = (j[:, None] >> jnp.arange(n-1, -1, -1)) & 1
    below = jnp.concatenate([jnp.zeros((2**n, 1), bits.dtype), jnp.cumsum(bits, 1)[:, :-1]], 1)
    jw = 1.0 - 2.0*(below % 2)
    return dict(flipped=j[:, None] ^ w, sign_c=jnp.where(bits == 1, jw, 0.0),
                sign_a=jnp.where(bits == 0, jw, 0.0))

def exact_determinant(C, tab):
    """|Psi> = prod_k (sum_i C_ik a_i^dag)|0> as a dense 2**n vector."""
    n, Nf = C.shape
    psi = jnp.zeros(2**n).at[0].set(1.0)
    for k in range(Nf-1, -1, -1):
        psi = (tab["sign_c"] * psi[tab["flipped"]]) @ C[:, k]
    return psi / jnp.linalg.norm(psi)

def sd_amplitude(C, bits):
    """<det|Psi_C> = det of the rows of C picked out by `bits`."""
    rows = np.where(np.asarray(bits) == 1)[0]
    return 0.0 if len(rows) != C.shape[1] else float(jnp.linalg.det(C[rows, :]))

def det_mps(bits):
    """|det> as a bond-dimension-1 pyblock3 MPS, labelled like to_pyblock3."""
    ts, nl = [], 0
    for o in bits:
        ts.append(SparseTensor(blocks=[SubTensor(data=jnp.ones((1, 1, 1)),
                                                 q_labels=(Q(nl), Q(int(o)), Q(nl + int(o))))]))
        nl += int(o)
    return MPS(tensors=ts)

def gauge_sign(U_rot, occ):
    """The +-1 relating the MPS to the determinant convention.

    The sweep produces a rotation R with R C spanning the unit vectors on the occupied
    sites, so |Psi_C> and the MPS differ by the determinant of that N_f x N_f change of
    basis -- which is exactly the rotated orbital matrix on the occupied rows, and is
    +-1 to machine precision.  Single-particle data only: no MPS needed, any n.
    """
    return int(round(float(np.linalg.det(U_rot[np.where(occ == 1)[0], :]))))

sgn = gauge_sign(U_rot, occ)
dets = [[(idx >> (n-1-j)) & 1 for j in range(n)] for idx in range(2**n)]
amp_pb3 = sgn * np.array([float(mps.dot(det_mps(d))) for d in dets])
amp_sd  = np.array([sd_amplitude(C, d) for d in dets])
print(f"gauge sign of this MPS: {sgn:+d}")
print(f"all {2**n} amplitudes, pyblock3 vs det C[occupied rows]:"
      f"  max |difference| = {np.abs(amp_pb3 - amp_sd).max():.2e}")

tab = jw_tables(n)
psi = to_dense(tensors); psi = psi / jnp.linalg.norm(psi)
A = tab["sign_a"] * psi[tab["flipped"]]                    # column i is a_i|psi>
print(f"correlation matrix off the MPS:  max |<a_i^dag a_j> - (C C^T)_ij| = "
      f"{jnp.abs(A.T @ A - Lam).max():.2e}")
print(f"against the dense 2**n determinant:  |<MPS|Psi_exact>| = "
      f"{abs(psi @ exact_determinant(C, tab)):.12f}")

gauge sign of this MPS: -1
all 1024 amplitudes, pyblock3 vs det C[occupied rows]:  max |difference| = 3.00e-15
correlation matrix off the MPS:  max |<a_i^dag a_j> - (C C^T)_ij| = 4.25e-15
against the dense 2**n determinant:  |<MPS|Psi_exact>| = 1.000000000000


In [8]:
#Checking that overlap agrees in different cases
def sd_overlap(A, B):
    """<Psi_A|Psi_B> = det(A^T B), or 0 when the particle numbers differ."""
    return 0.0 if A.shape[1] != B.shape[1] else float(jnp.linalg.det(A.T @ B))

def sd_to_pyblock3(C, chi_max=None, cutoff=1e-14):
    """orbitals -> GMPS -> symmetric MPS -> pyblock3, plus its gauge sign."""
    occ, gates, Bs, U_rot = gmps(C)
    tensors, qn = build(occ, gates, chi_max=chi_max, cutoff=cutoff)
    m = to_pyblock3(tensors, qn)
    return m, gauge_sign(U_rot, occ), max(Bs)

sds = {"SSH d=0.4": ssh_orbitals(n, 4, 0.4),
       "SSH d=1.2": ssh_orbitals(n, 4, 1.2),
       "uniform  ": ssh_orbitals(n, 4, 0.0),
       "random   ": jnp.linalg.qr(jax.random.normal(jax.random.PRNGKey(0), (n, 4)))[0],
       "SSH Nf=3 ": ssh_orbitals(n, 3, 0.4)}

built = {}
print("  determinant   Nf   max B   bond dims                   gauge")
for name, Cx in sds.items():
    m, s, Bmax_used = sd_to_pyblock3(Cx)
    built[name] = (Cx, m, s)
    print(f"  {name}      {Cx.shape[1]}     {Bmax_used:2d}     {m.show_bond_dims():27s} {s:+d}")

print("\n  overlap                    det(A^T B)         pyblock3 MPS     |difference|")
pairs = [("SSH d=0.4", "SSH d=0.4"), ("SSH d=0.4", "SSH d=1.2"),
         ("SSH d=0.4", "uniform  "), ("SSH d=1.2", "random   "),
         ("uniform  ", "random   "), ("SSH d=0.4", "SSH Nf=3 ")]
worst = 0.0
for a, b in pairs:
    (Ca, ma, sa), (Cb, mb, sb) = built[a], built[b]
    ref = sd_overlap(Ca, Cb)
    got = sa * sb * float(ma.dot(mb))
    worst = max(worst, abs(got - ref))
    print(f"  <{a}|{b}>     {ref:+.12f}   {got:+.12f}   {abs(got-ref):.1e}")
print(f"\n  worst disagreement: {worst:.1e}    (different fillings give exactly 0, by the bond"
      f" charges alone)")

  determinant   Nf   max B   bond dims                   gauge
  SSH d=0.4      4      5     1|2|4|8|16|16|16|8|4|2|1    -1
  SSH d=1.2      4      5     1|2|4|8|16|16|16|8|4|2|1    +1
  uniform        4      5     1|2|4|8|16|16|16|8|4|2|1    +1
  random         4      5     1|2|4|8|16|16|16|8|4|2|1    -1
  SSH Nf=3       3      4     1|2|4|8|8|8|8|8|4|2|1       +1

  overlap                    det(A^T B)         pyblock3 MPS     |difference|
  <SSH d=0.4|SSH d=0.4>     +1.000000000000   +1.000000000000   3.1e-15
  <SSH d=0.4|SSH d=1.2>     +0.960594613690   +0.960594613690   2.1e-15
  <SSH d=0.4|uniform  >     +0.973197470480   +0.973197470480   1.8e-15
  <SSH d=1.2|random   >     -0.081850098352   -0.081850098352   7.2e-16
  <uniform  |random   >     -0.075149149395   -0.075149149395   1.0e-15
  <SSH d=0.4|SSH Nf=3 >     +0.000000000000   -0.000000000000   0.0e+00

  worst disagreement: 3.1e-15    (different fillings give exactly 0, by the bond charges alone)


In [9]:
#Creating an MPO from the gates rather than applying the gates sequentially
def schedule_layers(gates):
    """Level the circuit into layers of mutually disjoint gates, preserving the order.

    A gate may only move earlier past gates it commutes with, i.e. disjoint ones, so
    level(g) = 1 + max(level of the gates sharing a site).  Same-level gates are disjoint.
    """
    level_of_site, layers = {}, []
    for p, theta in gates:
        lv = max(level_of_site.get(p, 0), level_of_site.get(p+1, 0))
        level_of_site[p] = level_of_site[p+1] = lv + 1
        if lv == len(layers):
            layers.append([])
        layers[lv].append((p, theta))
    return layers

def gate_mpo_pair(theta):
    """Split V_hat into two MPO tensors W[wl, out, in, wr] with definite bond charges."""
    g = np.asarray(V_hat(theta))                       # [out_p, out_q, in_p, in_q]
    Wl, Wr, charges = [], [], []
    for delta in (-1, 0, 1):
        rows = [(o, i) for o in (0, 1) for i in (0, 1) if o - i == delta]
        cols = [(o, i) for o in (0, 1) for i in (0, 1) if o - i == -delta]
        M = np.array([[g[ro, co, ri, ci] for (co, ci) in cols] for (ro, ri) in rows])
        if np.abs(M).max() == 0.0:
            continue
        u, s, vt = np.linalg.svd(M, full_matrices=False)
        keep = s > 1e-14 * max(s.max(), 1e-300)
        u, s, vt = u[:, keep], s[keep], vt[keep]
        for t in range(len(s)):
            wl, wr = np.zeros((2, 2)), np.zeros((2, 2))
            for (o, i), val in zip(rows, u[:, t] * np.sqrt(s[t])):
                wl[o, i] = val
            for (o, i), val in zip(cols, vt[t] * np.sqrt(s[t])):
                wr[o, i] = val
            Wl.append(wl); Wr.append(wr); charges.append(delta)
    r = len(charges)
    return (np.stack(Wl, -1).reshape(1, 2, 2, r),
            np.stack(Wr, 0).reshape(r, 2, 2, 1), np.array(charges, int))

def apply_layer_mpo(tensors, qn, layer, chi_max=None, cutoff=1e-12):
    """Contract one layer's MPO into the MPS, then sweep-compress back down."""
    n = len(tensors)
    W = [np.eye(2).reshape(1, 2, 2, 1) for _ in range(n)]
    qw = [np.zeros(1, int) for _ in range(n+1)]
    for p, theta in layer:
        W[p], W[p+1], qw[p+1] = gate_mpo_pair(theta)
    new, qnew = [], [np.zeros(1, int)]
    for i in range(n):
        T = jnp.einsum("wxyz,ayb->awxbz", jnp.asarray(W[i]), tensors[i])
        Dl, rl, _, Dr, rr = T.shape
        new.append(T.reshape(Dl*rl, 2, Dr*rr))
        qnew.append((qn[i+1][:, None] + qw[i+1][None, :]).ravel())
    for p in range(n-1):                               # one left-to-right compression sweep
        T = jnp.tensordot(new[p], new[p+1], axes=1)
        new[p], new[p+1], qnew[p+1] = split_two_site(T, qnew[p], qnew[p+2], chi_max, cutoff)
    return new, qnew

def build_by_layers(occ, gates, chi_max=None, cutoff=1e-12):
    tensors, qn = product_state(occ)
    for layer in schedule_layers(list(reversed(gates))):
        tensors, qn = apply_layer_mpo(tensors, qn, layer, chi_max, cutoff)
    return tensors, qn

# same state?
ts_l, qn_l = build_by_layers(occ, gates, cutoff=1e-14)
psi_g = to_dense(tensors); psi_l = to_dense(ts_l)
layers = schedule_layers(list(reversed(gates)))
print(f"{len(gates)} gates -> {len(layers)} layers"
      f" (sizes {[len(l) for l in layers]})")
print(f"  gate-by-gate bond dims {[t.shape[0] for t in tensors] + [tensors[-1].shape[2]]}")
print(f"  layer-MPO    bond dims {[t.shape[0] for t in ts_l] + [ts_l[-1].shape[2]]}")
print(f"  |<gate-by-gate|layer-MPO>| = "
      f"{abs(float(psi_g @ psi_l))/float(jnp.linalg.norm(psi_g)*jnp.linalg.norm(psi_l)):.12f}")

# and the cost, counting the SVDs each route performs
import time
_svd, CALLS = jnp.linalg.svd, []
def spy(M, **kw):
    CALLS.append(tuple(M.shape)); return _svd(M, **kw)

print("\n   n   Nf  gates layers |   gate-by-gate    |     layer-MPO     | same state")
print("                          |  time  svds shapes|  time  svds shapes|")
for n_c, Nf_c in [(10, 4), (16, 8), (24, 12)]:
    C_c = ssh_orbitals(n_c, Nf_c)
    o_c, g_c, _, _ = gmps(C_c)
    build(o_c, g_c, cutoff=1e-14); build_by_layers(o_c, g_c, cutoff=1e-14)   # warm caches
    jnp.linalg.svd = spy
    CALLS.clear(); t0 = time.perf_counter(); a, _ = build(o_c, g_c, cutoff=1e-14)
    tg, ng, sg = time.perf_counter()-t0, len(CALLS), len(set(CALLS))
    CALLS.clear(); t0 = time.perf_counter(); b, _ = build_by_layers(o_c, g_c, cutoff=1e-14)
    tl, nl, sl = time.perf_counter()-t0, len(CALLS), len(set(CALLS))
    jnp.linalg.svd = _svd
    pa, pb = to_dense(a), to_dense(b)
    fid = abs(float(pa @ pb))/float(jnp.linalg.norm(pa)*jnp.linalg.norm(pb))
    print(f"  {n_c:3d} {Nf_c:3d} {len(g_c):5d} {len(schedule_layers(list(reversed(g_c)))):5d}"
          f"  | {tg*1e3:5.0f}ms {ng:4d} {sg:5d}| {tl*1e3:5.0f}ms {nl:4d} {sl:5d}| {fid:.10f}")

24 gates -> 12 layers (sizes [1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 2, 1])
  gate-by-gate bond dims [1, 2, 4, 8, 16, 16, 16, 8, 4, 2, 1]
  layer-MPO    bond dims [1, 2, 4, 8, 16, 16, 16, 8, 4, 2, 1]
  |<gate-by-gate|layer-MPO>| = 1.000000000000

   n   Nf  gates layers |   gate-by-gate    |     layer-MPO     | same state
                          |  time  svds shapes|  time  svds shapes|
   10   4    24    12  |    69ms   74    15|   277ms  284    43| 1.0000000000
   16   8    58    20  |   209ms  221    24|   916ms  968    89| 1.0000000000
   24  12   114    29  |   495ms  499    33|  2436ms 2565   126| 1.0000000000


In [10]:
from pyblock3.algebra.core import SparseTensor as CoreSparse, SubTensor as CoreSub, FermionTensor
from pyblock3.algebra.mps import MPS as CoreMPS
#pyblock3 supplies the JW string itself, so the JW-resolved V(\theta) double-counts it: 
# the $\Delta=-1$ branch needs a $-1.
MPO_SIGN = {-1: -1.0, 0: 1.0, 1: 1.0}       # see point 3 above
#Native pyblock3 MPO compression
def _ferm(even, odd):
    return FermionTensor(odd=CoreSparse(blocks=odd) if odd else None,
                         even=CoreSparse(blocks=even) if even else None)

def _identity_site():
    return _ferm([CoreSub(reduced=np.ones((1,1,1,1)), q_labels=(Q(0), Q(o), Q(o), Q(0)))
                  for o in (0, 1)], [])

def gate_fermion_pair(theta):
    """V_hat as two FermionTensors, terms stacked along the MPO bond per charge sector."""
    g = np.asarray(V_hat(theta))
    left, right = ({0: {}, 1: {}}, {0: {}, 1: {}})
    for delta in (-1, 0, 1):
        rows = [(o, i) for o in (0, 1) for i in (0, 1) if o - i == delta]
        cols = [(o, i) for o in (0, 1) for i in (0, 1) if o - i == -delta]
        M = np.array([[g[ro, co, ri, ci] for (co, ci) in cols] for (ro, ri) in rows])
        if np.abs(M).max() == 0.0:
            continue
        u, s, vt = np.linalg.svd(M, full_matrices=False)
        nt = int((s > 1e-14).sum())
        for j, (o, i) in enumerate(rows):
            left[(o+i) % 2][(delta, o, i)] = MPO_SIGN[delta] * np.array(
                [u[j, t]*np.sqrt(s[t]) for t in range(nt)])
        for j, (o, i) in enumerate(cols):
            right[(o+i) % 2][(delta, o, i)] = np.array(
                [vt[t, j]*np.sqrt(s[t]) for t in range(nt)])
    def blocks(d, is_left):
        out = {0: [], 1: []}
        for par in (0, 1):
            for (delta, o, i), vec in d[par].items():
                if np.abs(vec).max() <= 1e-15:
                    continue
                sh = (1, 1, 1, len(vec)) if is_left else (len(vec), 1, 1, 1)
                ql = (Q(0), Q(o), Q(i), Q(delta)) if is_left else (Q(delta), Q(o), Q(i), Q(0))
                out[par].append(CoreSub(reduced=vec.reshape(sh), q_labels=ql))
        return out
    bl, br = blocks(left, True), blocks(right, False)
    return _ferm(bl[0], bl[1]), _ferm(br[0], br[1])

def layer_mpo_native(layer, n):
    ts = [_identity_site() for _ in range(n)]
    for p, theta in layer:
        ts[p], ts[p+1] = gate_fermion_pair(float(theta))
    return CoreMPS(tensors=ts)

def product_mps_native(occ):
    ts, q = [], 0
    for o in occ:
        ts.append(CoreSparse(blocks=[CoreSub(reduced=np.ones((1,1,1)),
                                             q_labels=(Q(q), Q(int(o)), Q(q+int(o))))]))
        q += int(o)
    return CoreMPS(tensors=ts)

def build_native(occ, gates, cutoff=1e-13):
    """Product state as a pyblock3 MPS, then one MPO per layer via pyblock3's own matmul."""
    mps = product_mps_native(occ)
    for layer in schedule_layers(list(reversed(gates))):
        r = layer_mpo_native(layer, len(occ)).__matmul__(mps).compress(cutoff=cutoff)
        mps = r[0] if isinstance(r, tuple) else r
    return mps

# correctness and cost against the hand-rolled routes
print("   n  Nf gates layers | hand gate-by-gate | hand layer-MPO | pyblock3 native MPO | fidelity")
for n_n, Nf_n in [(10, 4), (12, 5), (14, 7)]:
    C_n = ssh_orbitals(n_n, Nf_n)
    o_n, g_n, _, _ = gmps(C_n)
    nl = len(schedule_layers(list(reversed(g_n))))
    ts_n, _ = build(o_n, g_n, cutoff=1e-13)
    ref_n = to_dense(ts_n); ref_n = ref_n / jnp.linalg.norm(ref_n)
    t0 = time.perf_counter(); build(o_n, g_n, cutoff=1e-13); t_g = time.perf_counter()-t0
    t0 = time.perf_counter(); build_by_layers(o_n, g_n, cutoff=1e-13); t_l = time.perf_counter()-t0
    build_native(o_n, g_n)
    t0 = time.perf_counter(); m_n = build_native(o_n, g_n); t_n = time.perf_counter()-t0
    dets_n = [np.isin(np.arange(n_n), c).astype(int)
              for c in combinations(range(n_n), Nf_n)]
    a_nat = np.array([float(m_n.dot(product_mps_native([int(x) for x in d]))) for d in dets_n])
    a_ref = np.array([float(ref_n[int("".join(map(str, d)), 2)]) for d in dets_n])
    a_nat /= np.linalg.norm(a_nat); a_ref /= np.linalg.norm(a_ref)
    print(f"  {n_n:3d} {Nf_n:3d} {len(g_n):5d} {nl:5d}  |     {t_g*1e3:8.1f} ms  |"
          f"  {t_l*1e3:8.1f} ms  |      {t_n*1e3:8.1f} ms  | {abs(a_nat @ a_ref):.10f}")

   n  Nf gates layers | hand gate-by-gate | hand layer-MPO | pyblock3 native MPO | fidelity
   10   4    24    12  |         69.8 ms  |     277.9 ms  |          54.1 ms  | 1.0000000000
   12   5    35    15  |        116.1 ms  |     987.0 ms  |          87.3 ms  | 1.0000000000
   14   7    47    18  |        167.7 ms  |     705.6 ms  |         133.3 ms  | 1.0000000000


## 9. Making JAX earn its keep: freeze the plan, replay it

Everything above runs on `jnp`, but none of it is compiled, and that is not an oversight —
the shapes are data. Each sector-blocked split has its own size, and the bonds grow as gates
are applied, so `jit` would recompile constantly. Measured on one build of this size: 251 SVD
calls, **33 distinct shapes**, and on a cold run **96% of the time is XLA compilation** rather
than arithmetic.

There are two ways out, and they are worth separating because they behave very differently.

**Pad to fixed shapes.** Give every bond the same width $\chi$, do one dense $2\chi\times2\chi$
SVD per gate, and compile that single kernel once. Shapes never change, so compilation is paid
once and does not grow with $n$. The cost is wasted arithmetic: every SVD is full width even
where the true rank is 2. One trap, worth stating because it is silent — the truncation must
**mask**, not slice. Keeping the largest $\chi$ singular values *by count* loses real weight
here: these local splits are never canonicalised, so the matrix's singular values are not the
state's Schmidt values and its rank can exceed the physical one. (Getting that wrong cost a
fidelity of 0.81 where 1.0 was expected, and raising $\chi$ fourfold did not fix it.) Zeroing
below a cutoff while keeping all $\chi$ columns is exact and equally static.

**Freeze every discrete decision — the plan/replay split.** Run the build once on the host and
record what it chose: the sector row/column lists and how many singular values survived, hence
every array shape. Then the replay takes only the angles and has nothing left to decide, so the
whole build traces to *one* XLA program. This is legitimate because the discrete choices are
locally constant in the angles: nudge $\theta$ and the same sectors and ranks come out, so the
frozen plan is exact in a neighbourhood — and it is the only version that `jit` and `grad` can
digest. Re-plan when the state moves far enough to change a rank.

In [11]:
def record_plan(occ, gates, chi_max=None, cutoff=1e-12):
    """Run the build once on the host, recording every shape it chooses.

    Returns (plan, qn): the plan is per-gate (p, Dl, Dr, sectors), each sector being
    (row indices, column indices, retained rank) -- all the shape information the replay
    needs.  qn is the final set of bond charges, needed for `to_pyblock3`.
    """
    n = len(occ)
    ts = [np.eye(2)[int(o)].reshape(1, 2, 1) for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    plan = []
    for p, theta in reversed(gates):
        g = np.asarray(V_hat(theta))
        T = np.einsum("xypq,apqb->axyb", g, np.tensordot(ts[p], ts[p+1], axes=1))
        Dl, _, _, Dr = T.shape
        M = T.reshape(Dl*2, 2*Dr)
        rc = (qn[p][:, None] + np.arange(2)[None, :]).ravel()
        cc = (qn[p+2][None, :] - np.arange(2)[:, None]).ravel()
        secs, A, Bc, qm = [], [], [], []
        for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
            r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
            u, s, vt = np.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
            k = int((s > cutoff * max(s.max(), 1e-300)).sum())
            if chi_max is not None:
                k = min(k, chi_max)
            if k == 0:
                continue
            secs.append((tuple(r), tuple(c), k))
            a = np.zeros((Dl*2, k)); a[r] = u[:, :k]
            b = np.zeros((k, 2*Dr)); b[:, c] = s[:k, None] * vt[:k]
            A.append(a); Bc.append(b); qm += [nm] * k
        plan.append((p, Dl, Dr, tuple(secs)))
        ts[p] = np.hstack(A).reshape(Dl, 2, -1)
        ts[p+1] = np.vstack(Bc).reshape(-1, 2, Dr)
        qn[p+1] = np.array(qm, int)
    return plan, qn

def make_replay(occ, plan):
    """Compile the build for a frozen plan: angles in, MPS tensors out, all shapes static."""
    def run(thetas):
        ts = [jax.nn.one_hot(int(o), 2).reshape(1, 2, 1) for o in occ]
        for (p, Dl, Dr, secs), theta in zip(plan, thetas):
            T = jnp.einsum("xypq,apqb->axyb", V_hat(theta),
                           jnp.tensordot(ts[p], ts[p+1], axes=1))
            M = T.reshape(Dl*2, 2*Dr)
            A, Bc = [], []
            for r, c, k in secs:
                r, c = np.array(r), np.array(c)
                u, s, vt = jnp.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
                A.append(jnp.zeros((Dl*2, k)).at[r].set(u[:, :k]))
                Bc.append(jnp.zeros((k, 2*Dr)).at[:, c].set(s[:k, None] * vt[:k]))
            ts = list(ts)
            ts[p] = jnp.concatenate(A, axis=1).reshape(Dl, 2, -1)
            ts[p+1] = jnp.concatenate(Bc, axis=0).reshape(-1, 2, Dr)
        return ts
    return jax.jit(run)

thetas = jnp.array([float(t) for _, t in reversed(gates)])
plan, qn_plan = record_plan(occ, gates, cutoff=1e-14)
replay = make_replay(occ, plan)

t0 = time.perf_counter(); ts_r = jax.block_until_ready(replay(thetas)); t_cold = time.perf_counter()-t0
t0 = time.perf_counter(); ts_r = jax.block_until_ready(replay(thetas)); t_warm = time.perf_counter()-t0
psi_r = to_dense(ts_r); psi_b = to_dense(tensors)
print(f"plan: {len(plan)} gates, {sum(len(s[3]) for s in plan)} sector splits,"
      f" bond charges identical to the dynamic build:"
      f" {all(np.array_equal(a, b) for a, b in zip(qn_plan, qn))}")
print(f"replay gives the same state: |<replay|dynamic>| = "
      f"{abs(float(psi_r @ psi_b))/float(jnp.linalg.norm(psi_r)*jnp.linalg.norm(psi_b)):.12f}")
print(f"  compile {t_cold*1e3:7.1f} ms   then {t_warm*1e3:6.2f} ms per call")

plan: 24 gates, 74 sector splits, bond charges identical to the dynamic build: True
replay gives the same state: |<replay|dynamic>| = 1.000000000000
  compile   712.0 ms   then   0.58 ms per call


In [12]:
# the same three strategies, timed side by side on a few sizes
def build_padded(occ, thetas, ps, chi, cutoff=1e-13):
    """Strategy 2: every bond padded to chi, one compiled kernel, truncation by MASK."""
    @partial(jax.jit, static_argnames=("chi",))
    def gate(Al, Ar, theta, chi):
        T = jnp.einsum("xypq,apqb->axyb", V_hat(theta), jnp.tensordot(Al, Ar, axes=1))
        u, s, vt = jnp.linalg.svd(T.reshape(2*chi, 2*chi), full_matrices=False)
        live = s > cutoff * jnp.maximum(s[0], 1e-300)
        s = jnp.where(live, s, 0.0)
        return ((u * live[None, :])[:, :chi].reshape(chi, 2, chi),
                (s[:chi, None] * vt[:chi]).reshape(chi, 2, chi))
    ts = [jnp.zeros((chi, 2, chi)).at[0, int(o), 0].set(1.0) for o in occ]
    for p, th in zip(ps, thetas):
        ts[p], ts[p+1] = gate(ts[p], ts[p+1], th, chi)
    return ts

def to_dense_padded(ts):
    """Contract a padded MPS: only index 0 of each boundary bond is live."""
    psi = ts[0][0]
    for A in ts[1:]:
        psi = jnp.tensordot(psi, A, axes=([-1], [0]))
    return psi[..., 0].reshape(-1)

print("\n   n  Nf gates  chi | dynamic jnp |  padded: warm / compile |  replay: warm / compile"
      " | fidelity pad/replay")
for n_s, Nf_s in [(10, 4), (14, 7), (18, 9)]:
    C_s = ssh_orbitals(n_s, Nf_s)
    o_s, g_s, _, _ = gmps(C_s)
    th_s = jnp.array([float(t) for _, t in reversed(g_s)])
    ps_s = np.array([p for p, _ in reversed(g_s)])
    ts_s, _ = build(o_s, g_s, cutoff=1e-14)
    chi_s = max([t.shape[0] for t in ts_s] + [ts_s[-1].shape[2]])
    ref_s = to_dense(ts_s); ref_s = ref_s / jnp.linalg.norm(ref_s)

    t0 = time.perf_counter(); build(o_s, g_s, cutoff=1e-14); t_dyn = time.perf_counter()-t0
    t0 = time.perf_counter(); build_padded(o_s, th_s, ps_s, chi_s); t_pc = time.perf_counter()-t0
    t0 = time.perf_counter(); tp = build_padded(o_s, th_s, ps_s, chi_s); t_pw = time.perf_counter()-t0
    pl_s, _ = record_plan(o_s, g_s, cutoff=1e-14)
    rp = make_replay(o_s, pl_s)
    t0 = time.perf_counter(); jax.block_until_ready(rp(th_s)); t_rc = time.perf_counter()-t0
    t0 = time.perf_counter(); tr = jax.block_until_ready(rp(th_s)); t_rw = time.perf_counter()-t0
    fid = lambda v: abs(float(v @ ref_s))/float(jnp.linalg.norm(v))
    print(f"  {n_s:3d} {Nf_s:3d} {len(g_s):5d} {chi_s:4d} | {t_dyn*1e3:8.0f} ms  |"
          f" {t_pw*1e3:8.1f} / {t_pc*1e3:6.0f} ms |"
          f" {t_rw*1e3:6.2f} / {t_rc*1e3:6.0f} ms |"
          f"  {fid(to_dense_padded(tp)):.8f} / {fid(to_dense(tr)):.8f}")


   n  Nf gates  chi | dynamic jnp |  padded: warm / compile |  replay: warm / compile | fidelity pad/replay
   10   4    24   16 |       79 ms  |    127.4 /    223 ms |   0.48 /    640 ms |  1.00000000 / 1.00000000
   14   7    47   64 |      185 ms  |     94.8 /    189 ms |   1.43 /   1700 ms |  1.00000000 / 1.00000000
   18   9    75  128 |      306 ms  |    442.8 /    422 ms |   3.99 /   2574 ms |  1.00000000 / 1.00000000
